# 프로젝트 9: 모델 바꿔보기 — StackedHourglass vs SimpleBaseline

In [1]:
# ── 공통 import & 경로 설정 ────────────────────────────────
import io, json, os, math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

PROJECT_PATH = os.path.join(os.getenv("HOME"), 'work/mpii')
IMAGE_PATH   = os.path.join(PROJECT_PATH, 'images')
MODEL_PATH   = os.path.join(PROJECT_PATH, 'models')
TRAIN_JSON   = os.path.join(PROJECT_PATH, 'mpii_human_pose_v1_u12_2', 'train.json')
VALID_JSON   = os.path.join(PROJECT_PATH, 'mpii_human_pose_v1_u12_2', 'validation.json')

os.makedirs(MODEL_PATH, exist_ok=True)
print('슝=3')

슝=3


In [2]:
# ── annotation 파싱 ────────────────────────────────────────
def parse_one_annotation(anno, image_dir):
    return {
        'filename'          : anno['image'],
        'filepath'          : os.path.join(image_dir, anno['image']),
        'joints_visibility' : anno['joints_vis'],
        'joints'            : anno['joints'],
        'center'            : anno['center'],
        'scale'             : anno['scale']
    }
print('슝=3')

슝=3


In [3]:
# ── MPIIDataset ────────────────────────────────────────────
from torch.utils.data import Dataset

class MPIIDataset(Dataset):
    def __init__(self, annotation_file, image_dir, transform=None):
        self.transform = transform
        with open(annotation_file, 'r') as f:
            annotations = json.load(f)
        self.annotations = [parse_one_annotation(a, image_dir) for a in annotations]

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):
        anno  = self.annotations[idx]
        image = Image.open(anno['filepath']).convert('RGB')
        if self.transform:
            image, heatmaps = self.transform({'image': image, 'annotation': anno})
            return image, heatmaps
        return image, anno

print('슝=3')

슝=3


In [4]:
# ── Preprocessor ───────────────────────────────────────────
class Preprocessor(object):
    def __init__(self, image_shape=(256,256,3), heatmap_shape=(64,64,16), is_train=False):
        self.is_train      = is_train
        self.image_shape   = image_shape
        self.heatmap_shape = heatmap_shape

    def __call__(self, example):
        features = self._parse(example)
        image    = Image.open(io.BytesIO(features['image/encoded']))
        margin   = torch.empty(1).uniform_(0.1, 0.3).item() if self.is_train else 0.2
        image, kp_x, kp_y = self._crop_roi(image, features, margin)
        image    = image.resize((self.image_shape[1], self.image_shape[0]))
        img_np   = np.array(image).astype(np.float32) / 127.5 - 1.0
        img_t    = torch.from_numpy(img_np).permute(2, 0, 1)
        heatmaps = self._make_heatmaps(features, kp_x, kp_y, self.heatmap_shape)
        return img_t, heatmaps

    def _parse(self, example):
        anno = example['annotation']
        buf  = io.BytesIO()
        example['image'].save(buf, format='JPEG')
        joints = anno['joints']
        return {
            'image/encoded'        : buf.getvalue(),
            'image/object/parts/x' : [j[0] for j in joints],
            'image/object/parts/y' : [j[1] for j in joints],
            'image/object/parts/v' : anno.get('joints_visibility', [1]*len(joints)),
            'image/object/center/x': anno['center'][0],
            'image/object/center/y': anno['center'][1],
            'image/object/scale'   : anno['scale'],
        }

    def _crop_roi(self, image, features, margin=0.2):
        w, h   = image.size
        kp_x   = torch.tensor(features['image/object/parts/x'], dtype=torch.int32)
        kp_y   = torch.tensor(features['image/object/parts/y'], dtype=torch.int32)
        body_h = features['image/object/scale'] * 200.0
        mx_x   = kp_x[kp_x > 0];  mx_y = kp_y[kp_y > 0]
        extra  = int(body_h * margin)
        xmin   = max(int(mx_x.min()) - extra, 0)
        ymin   = max(int(mx_y.min()) - extra, 0)
        xmax   = min(int(mx_x.max()) + extra, w)
        ymax   = min(int(mx_y.max()) + extra, h)
        crop   = image.crop((xmin, ymin, xmax, ymax))
        nw, nh = xmax - xmin, ymax - ymin
        eff_x  = (kp_x.float() - xmin) / nw
        eff_y  = (kp_y.float() - ymin) / nh
        return crop, eff_x, eff_y

    def _gaussian(self, height, width, y0, x0, visibility=2, sigma=1, scale=12):
        heatmap = torch.zeros((height, width), dtype=torch.float32)
        xmin, ymin = x0-3*sigma, y0-3*sigma
        xmax, ymax = x0+3*sigma, y0+3*sigma
        if xmin >= width or ymin >= height or xmax < 0 or ymax < 0 or visibility == 0:
            return heatmap
        size = int(6*sigma + 1)
        r    = torch.arange(0, size, dtype=torch.float32)
        xg, yg = torch.meshgrid(r, r, indexing='xy')
        cx, cy = size//2, size//2
        patch  = torch.exp(-(((xg-cx)**2 + (yg-cy)**2) / (sigma**2*2))) * scale
        px0, py0 = max(0, -xmin), max(0, -ymin)
        px1, py1 = min(xmax, width)-xmin, min(ymax, height)-ymin
        hx0, hy0 = max(0, xmin), max(0, ymin)
        hx1, hy1 = min(xmax, width), min(ymax, height)
        heatmap[hy0:hy1, hx0:hx1] = patch[int(py0):int(py1), int(px0):int(px1)]
        return heatmap

    def _make_heatmaps(self, features, kp_x, kp_y, heatmap_shape):
        v  = torch.tensor(features['image/object/parts/v'], dtype=torch.float32)
        x  = torch.round(torch.tensor(kp_x, dtype=torch.float32) * heatmap_shape[0]).int()
        y  = torch.round(torch.tensor(kp_y, dtype=torch.float32) * heatmap_shape[1]).int()
        hms = [self._gaussian(heatmap_shape[1], heatmap_shape[0],
                              int(y[i]), int(x[i]), int(v[i]))
               for i in range(heatmap_shape[2])]
        return torch.stack(hms, dim=0).permute(1, 2, 0)   # (H, W, C)

print('슝=3')

슝=3


In [5]:
# ── StackedHourglass 모델 (기존 실습 코드 그대로) ────────────
class BottleneckBlock(nn.Module):
    def __init__(self, in_channels, filters, stride=1, downsample=False):
        super().__init__()
        self.downsample = downsample
        if downsample:
            self.down_conv = nn.Conv2d(in_channels, filters, 1, stride=stride, bias=False)
        self.bn1   = nn.BatchNorm2d(in_channels, momentum=0.9)
        self.relu  = nn.ReLU(inplace=True)
        self.conv1 = nn.Conv2d(in_channels, filters//2, 1, bias=False)
        self.bn2   = nn.BatchNorm2d(filters//2, momentum=0.9)
        self.conv2 = nn.Conv2d(filters//2, filters//2, 3, stride=stride, padding=1, bias=False)
        self.bn3   = nn.BatchNorm2d(filters//2, momentum=0.9)
        self.conv3 = nn.Conv2d(filters//2, filters, 1, bias=False)

    def forward(self, x):
        identity = self.down_conv(x) if self.downsample else x
        out = self.conv1(self.relu(self.bn1(x)))
        out = self.conv2(self.relu(self.bn2(out)))
        out = self.conv3(self.relu(self.bn3(out)))
        return out + identity

class HourglassModule(nn.Module):
    def __init__(self, order, filters, num_residual):
        super().__init__()
        self.order     = order
        self.up1_0     = BottleneckBlock(filters, filters)
        self.up1_extra = nn.Sequential(*[BottleneckBlock(filters, filters) for _ in range(num_residual)])
        self.pool      = nn.MaxPool2d(2, 2)
        self.low1      = nn.Sequential(*[BottleneckBlock(filters, filters) for _ in range(num_residual)])
        self.low2      = HourglassModule(order-1, filters, num_residual) if order > 1 \
                         else nn.Sequential(*[BottleneckBlock(filters, filters) for _ in range(num_residual)])
        self.low3      = nn.Sequential(*[BottleneckBlock(filters, filters) for _ in range(num_residual)])
        self.up        = nn.Upsample(scale_factor=2, mode='nearest')

    def forward(self, x):
        up  = self.up1_extra(self.up1_0(x))
        low = self.low3(self.low2(self.low1(self.pool(x))))
        return up + self.up(low)

class LinearLayer(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, 1, bias=False)
        self.bn   = nn.BatchNorm2d(out_ch, momentum=0.9)
        self.relu = nn.ReLU(inplace=True)
        nn.init.kaiming_normal_(self.conv.weight, mode='fan_out', nonlinearity='relu')

    def forward(self, x):
        return self.relu(self.bn(self.conv(x)))

class StackedHourglassNetwork(nn.Module):
    def __init__(self, input_shape=(256,256,3), num_stack=4, num_residual=1, num_heatmap=16):
        super().__init__()
        self.num_stack = num_stack
        self.conv1 = nn.Conv2d(3, 64, 7, stride=2, padding=3, bias=False)
        self.bn1   = nn.BatchNorm2d(64, momentum=0.9)
        self.relu  = nn.ReLU(inplace=True)
        self.b1    = BottleneckBlock(64,  128, downsample=True)
        self.pool  = nn.MaxPool2d(2, 2)
        self.b2    = BottleneckBlock(128, 128)
        self.b3    = BottleneckBlock(128, 256, downsample=True)
        self.hg = nn.ModuleList()
        self.res= nn.ModuleList()
        self.lin= nn.ModuleList()
        self.hm = nn.ModuleList()
        self.i1 = nn.ModuleList()
        self.i2 = nn.ModuleList()
        for i in range(num_stack):
            self.hg.append(HourglassModule(4, 256, num_residual))
            self.res.append(nn.Sequential(*[BottleneckBlock(256,256) for _ in range(num_residual)]))
            self.lin.append(LinearLayer(256, 256))
            self.hm.append(nn.Conv2d(256, num_heatmap, 1))
            if i < num_stack - 1:
                self.i1.append(nn.Conv2d(256, 256, 1))
                self.i2.append(nn.Conv2d(num_heatmap, 256, 1))

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.pool(self.b1(x))
        x = self.b3(self.b2(x))
        outputs = []
        for i in range(self.num_stack):
            lin = self.lin[i](self.res[i](self.hg[i](x)))
            hm  = self.hm[i](lin)
            outputs.append(hm)
            if i < self.num_stack - 1:
                x = self.i1[i](lin) + self.i2[i](hm)
        return outputs

print('슝=3')

슝=3


In [6]:
# ── Trainer (기존 실습 코드 그대로) ───────────────────────────
import torch.optim as optim

class Trainer(object):
    def __init__(self, model, epochs, global_batch_size, initial_learning_rate):
        self.model             = model
        self.epochs            = epochs
        self.global_batch_size = global_batch_size
        self.loss_object       = nn.MSELoss(reduction='none')
        self.optimizer         = optim.Adam(self.model.parameters(), lr=initial_learning_rate)
        self.current_learning_rate = initial_learning_rate
        self.last_val_loss     = math.inf
        self.lowest_val_loss   = math.inf
        self.patience_count    = 0
        self.max_patience      = 10
        self.best_model        = None
        # 비교용 이력 저장 (프로젝트 추가)
        self.train_losses      = []
        self.val_losses        = []
        if torch.cuda.device_count() > 1:
            print(f"멀티 GPU 사용 (GPU 개수: {torch.cuda.device_count()})")
            self.model = nn.DataParallel(self.model)
        else:
            print("단일 GPU 혹은 CPU 사용")

    def lr_decay(self):
        if self.patience_count >= self.max_patience:
            self.current_learning_rate /= 10.0
            self.patience_count = 0
        elif self.last_val_loss == self.lowest_val_loss:
            self.patience_count = 0
        self.patience_count += 1
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = self.current_learning_rate

    def compute_loss(self, labels, outputs):
        # labels: (B, H, W, C) → (B, C, H, W)
        labels = labels.permute(0, 3, 1, 2)
        loss = 0
        for output in outputs:
            weights      = (labels > 0).float() * 81 + 1
            squared_error = (labels - output) ** 2
            weighted_error = squared_error * weights
            loss += weighted_error.mean() / self.global_batch_size
        return loss

    def train_step(self, images, labels, device):
        self.model.train()
        images = images.to(device);  labels = labels.to(device)
        self.optimizer.zero_grad()
        outputs = self.model(images)
        loss = self.compute_loss(labels, outputs)
        loss.backward()
        self.optimizer.step()
        return loss.item()

    def val_step(self, images, labels, device):
        self.model.eval()
        with torch.no_grad():
            images = images.to(device);  labels = labels.to(device)
            outputs = self.model(images)
            loss = self.compute_loss(labels, outputs)
        return loss.item()

    def run(self, train_loader, val_loader, device):
        for epoch in range(1, self.epochs + 1):
            self.lr_decay()
            print(f"Start epoch {epoch} with learning rate {self.current_learning_rate:.6f}")
            total_train_loss = 0.0;  num_train_batches = 0
            for images, labels in train_loader:
                batch_loss = self.train_step(images, labels, device)
                total_train_loss += batch_loss;  num_train_batches += 1
                print(f"[Train] batch {num_train_batches} loss {batch_loss:.4f} "
                      f"avg_loss {total_train_loss/num_train_batches:.4f}")
            train_loss = total_train_loss / num_train_batches
            self.train_losses.append(train_loss)
            print(f"Epoch {epoch} train loss {train_loss:.4f}")
            total_val_loss = 0.0;  num_val_batches = 0
            for images, labels in val_loader:
                batch_loss = self.val_step(images, labels, device)
                num_val_batches += 1
                print(f"[Val] batch {num_val_batches} loss {batch_loss:.4f}")
                if not math.isnan(batch_loss):
                    total_val_loss += batch_loss
                else:
                    num_val_batches -= 1
            val_loss = total_val_loss / num_val_batches if num_val_batches > 0 else float('nan')
            self.val_losses.append(val_loss)
            print(f"Epoch {epoch} val loss {val_loss:.4f}")
            if val_loss < self.lowest_val_loss:
                self.save_model(epoch, val_loss)
                self.lowest_val_loss = val_loss
            self.last_val_loss = val_loss
        return self.best_model

    def save_model(self, epoch, loss):
        model_name = os.path.join(MODEL_PATH, f'model-epoch-{epoch}-loss-{loss:.4f}.pt')
        torch.save(self.model.state_dict(), model_name)
        self.best_model = model_name
        print(f"Model {model_name} saved.")

print('슝=3')

슝=3


In [7]:
# ── DataLoader 생성 함수 (기존 실습 코드 그대로) ───────────────
from torch.utils.data import DataLoader

IMAGE_SHAPE  = (256, 256, 3)
HEATMAP_SIZE = (64, 64)

def create_dataloader(annotation_file, image_dir, batch_size, num_heatmap, is_train=True):
    preprocess = Preprocessor(
        image_shape=IMAGE_SHAPE,
        heatmap_shape=(HEATMAP_SIZE[0], HEATMAP_SIZE[1], num_heatmap),
        is_train=is_train
    )
    dataset = MPIIDataset(annotation_file, image_dir, transform=preprocess)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=is_train,
        num_workers=4,
        pin_memory=True,
        drop_last=False,
        prefetch_factor=2
    )

print('슝=3')

슝=3


In [8]:
# ── 키포인트 추출 / 시각화 함수 (기존 실습 코드 그대로) ─────────
import torchvision.transforms as transforms

MPII_BONES = [
    (0,1),(1,2),(2,6),(3,6),(3,4),(4,5),
    (6,7),(7,8),(8,9),
    (7,12),(12,11),(11,10),
    (7,13),(13,14),(14,15),
]

def find_max_coordinates(heatmaps):
    H, W, C  = heatmaps.shape
    flat     = heatmaps.reshape(-1, C)
    indices  = torch.argmax(flat, dim=0)
    y = indices // W;  x = indices % W
    return torch.stack([x, y], dim=1)

def extract_keypoints_from_heatmap(heatmaps):
    H, W, C   = heatmaps.shape
    max_kps   = find_max_coordinates(heatmaps)
    padded    = F.pad(heatmaps.permute(2,0,1), (1,1,1,1)).permute(1,2,0)
    adjusted  = []
    for i, kp in enumerate(max_kps):
        mx, my  = int(kp[0]) + 1, int(kp[1]) + 1
        patch   = padded[my-1:my+2, mx-1:mx+2, i].clone()
        patch[1,1] = 0
        idx     = torch.argmax(patch.reshape(-1)).item()
        ny, nx  = idx // 3, idx % 3
        adjusted.append((kp[0].item() + (nx-1)/4.0,
                         kp[1].item() + (ny-1)/4.0))
    kps = torch.clamp(torch.tensor(adjusted), 0, H)
    return kps / H

def predict(model, image_path):
    image = Image.open(image_path).convert('RGB')
    pre   = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.ToTensor(),
        transforms.Lambda(lambda x: x * 2 - 1)
    ])
    inp = pre(image).unsqueeze(0).to(next(model.parameters()).device)
    model.eval()
    with torch.no_grad():
        outputs = model(inp)
    if not isinstance(outputs, list):
        outputs = [outputs]
    hm  = outputs[-1].squeeze(0).permute(1, 2, 0).detach().cpu()
    kps = extract_keypoints_from_heatmap(hm)
    return image, kps

def draw_skeleton_on_image(image, keypoints):
    img_np = np.array(image) if not isinstance(image, np.ndarray) else image
    fig, ax = plt.subplots(1)
    ax.imshow(img_np)
    joints = [(kp[0].item() * img_np.shape[1],
               kp[1].item() * img_np.shape[0]) for kp in keypoints]
    for bone in MPII_BONES:
        j1, j2 = joints[bone[0]], joints[bone[1]]
        ax.plot([j1[0], j2[0]], [j1[1], j2[1]], linewidth=5, alpha=0.7)
    for jx, jy in joints:
        ax.scatter(jx, jy, s=40, c='white', edgecolors='black', lw=1.5, zorder=5)
    plt.show()

print('슝=3')

슝=3


## STEP 1 — SimpleBaseline 모델 완성하기



In [9]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 1 : SimpleBaseline 모델 구현
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#
# 논문: Simple Baselines for Human Pose Estimation (Xiao et al., ECCV 2018)
#
# 핵심 구조
# ──────────────────────────────────────────────────────────
#  입력 (B, 3, 256, 256)
#       ↓
#  [ResNet-50 Backbone]   ← ImageNet pretrained 사용
#   conv1 → bn → relu → maxpool
#   layer1 → layer2 → layer3 → layer4
#   출력: (B, 2048, 8, 8)   ← stride=32이라 256/32=8
#       ↓
#  [Deconvolution Head]   ← 이 부분이 핵심 아이디어
#   ConvTranspose2d(2048→256, k=4, stride=2)  8  → 16
#   ConvTranspose2d(256→256,  k=4, stride=2)  16 → 32
#   ConvTranspose2d(256→256,  k=4, stride=2)  32 → 64
#       ↓
#  [Final Conv]
#   Conv2d(256 → num_heatmap, k=1)
#   출력: (B, 16, 64, 64)   ← HG와 동일한 shape
#
# StackedHourglass와 무엇이 다른가?
#   HG : skip connection 있는 재귀 구조, 처음부터 학습
#   SB : ImageNet pretrained ResNet + 간단한 deconv head
#        → 사전학습 덕분에 적은 epoch에서도 빠른 수렴
#        → 구조가 단순해서 코드도 짧고 이해하기 쉬움
# ──────────────────────────────────────────────────────────

import torchvision.models as tvm

class SimpleBaseline(nn.Module):
    def __init__(self, num_heatmap=16, pretrained=True):
        super(SimpleBaseline, self).__init__()

        # ── 1. Backbone: ResNet-50 ──────────────────────────────
        # torchvision의 ResNet-50을 불러온 뒤
        # 마지막 avgpool과 fc(분류 레이어)를 제거합니다.
        # layer4 출력인 2048채널 feature map을 backbone 출력으로 사용합니다.
        backbone = tvm.resnet50(
            weights=tvm.ResNet50_Weights.DEFAULT if pretrained else None
        )
        self.backbone = nn.Sequential(
            backbone.conv1,    # (B, 3,    256, 256) → (B, 64,   128, 128)
            backbone.bn1,
            backbone.relu,
            backbone.maxpool,  # (B, 64,   128, 128) → (B, 64,    64,  64)
            backbone.layer1,   # (B, 64,    64,  64) → (B, 256,   64,  64)
            backbone.layer2,   # (B, 256,   64,  64) → (B, 512,   32,  32)
            backbone.layer3,   # (B, 512,   32,  32) → (B, 1024,  16,  16)
            backbone.layer4,   # (B, 1024,  16,  16) → (B, 2048,   8,   8)
        )

        # ── 2. Deconvolution Head ───────────────────────────────
        # ConvTranspose2d(kernel=4, stride=2, padding=1)의 출력 크기 공식:
        #   output = (input - 1) × stride - 2 × padding + kernel
        #          = (input - 1) × 2 - 2 × 1 + 4
        #          = input × 2
        # → 정확히 2배로 해상도 증가
        #
        # 각 단계마다 BatchNorm + ReLU를 붙이는 이유:
        #   학습 안정성 + 비선형성 추가
        self.deconv_head = nn.Sequential(
            # 단계 1: (B, 2048, 8,  8)  → (B, 256, 16, 16)
            nn.ConvTranspose2d(2048, 256, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(256, momentum=0.9),
            nn.ReLU(inplace=True),
            # 단계 2: (B, 256,  16, 16) → (B, 256, 32, 32)
            nn.ConvTranspose2d(256, 256, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(256, momentum=0.9),
            nn.ReLU(inplace=True),
            # 단계 3: (B, 256,  32, 32) → (B, 256, 64, 64)
            nn.ConvTranspose2d(256, 256, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(256, momentum=0.9),
            nn.ReLU(inplace=True),
        )

        # ── 3. 최종 heatmap 출력 ─────────────────────────────────
        # 1×1 conv로 채널 수를 관절 수(16)에 맞춤
        # 출력: (B, num_heatmap, 64, 64)
        self.final_conv = nn.Conv2d(256, num_heatmap, kernel_size=1)

        # ── 4. Deconv Head 가중치 초기화 ────────────────────────
        # backbone(ResNet)은 pretrained 가중치를 그대로 사용
        # 새로 추가한 deconv head는 작은 표준편차(0.001)로 초기화
        # → 학습 초기에 예측값이 0에 가깝게 시작 → 안정적인 학습
        self._init_weights()

    def _init_weights(self):
        for m in self.deconv_head.modules():
            if isinstance(m, nn.ConvTranspose2d):
                nn.init.normal_(m.weight, std=0.001)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
        nn.init.normal_(self.final_conv.weight, std=0.001)
        nn.init.constant_(self.final_conv.bias, 0)

    def forward(self, x):
        # x: (B, 3, 256, 256)
        x = self.backbone(x)        # → (B, 2048, 8, 8)
        x = self.deconv_head(x)     # → (B, 256, 64, 64)
        x = self.final_conv(x)      # → (B, num_heatmap, 64, 64)
        # StackedHourglass는 list를 반환하므로 인터페이스를 통일
        # → Trainer의 compute_loss, predict 함수를 수정 없이 그대로 사용 가능
        return [x]

print('슝=3')

슝=3


In [10]:
# ── STEP 1 검증: 모델이 올바르게 구현되었는지 숫자로 확인 ──────
device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dummy   = torch.randn(2, 3, 256, 256).to(device)

sb_check = SimpleBaseline(num_heatmap=16, pretrained=False).to(device)
hg_check = StackedHourglassNetwork(num_stack=4, num_residual=1, num_heatmap=16).to(device)

sb_out = sb_check(dummy)
hg_out = hg_check(dummy)

print("=" * 55)
print("  STEP 1 구현 검증")
print("=" * 55)
print(f"입력 shape              : {list(dummy.shape)}")
print()
print(f"[SimpleBaseline]")
print(f"  출력 리스트 길이        : {len(sb_out)}  (HG와 인터페이스 통일 확인)")
print(f"  출력 shape             : {list(sb_out[-1].shape)}")
print(f"  기대값                 : [2, 16, 64, 64]  ← 일치하면 OK")
print()
print(f"[StackedHourglass]")
print(f"  출력 리스트 길이        : {len(hg_out)}  (num_stack=4이므로 4개)")
print(f"  출력 shape (마지막)    : {list(hg_out[-1].shape)}")
print()

hg_params = sum(p.numel() for p in hg_check.parameters())
sb_params = sum(p.numel() for p in sb_check.parameters())
print(f"[파라미터 수 비교]")
print(f"  StackedHourglass      : {hg_params:>12,}  ({hg_params/1e6:.1f}M)")
print(f"  SimpleBaseline        : {sb_params:>12,}  ({sb_params/1e6:.1f}M)")
print()
print("✅ 두 모델의 출력 shape이 동일 → STEP 2에서 한 줄만 바꾸면 됩니다")
print("=" * 55)

del sb_check, hg_check, dummy
torch.cuda.empty_cache()

  STEP 1 구현 검증
입력 shape              : [2, 3, 256, 256]

[SimpleBaseline]
  출력 리스트 길이        : 1  (HG와 인터페이스 통일 확인)
  출력 shape             : [2, 16, 64, 64]
  기대값                 : [2, 16, 64, 64]  ← 일치하면 OK

[StackedHourglass]
  출력 리스트 길이        : 4  (num_stack=4이므로 4개)
  출력 shape (마지막)    : [2, 16, 64, 64]

[파라미터 수 비교]
  StackedHourglass      :   16,251,392  (16.3M)
  SimpleBaseline        :   33,999,440  (34.0M)

✅ 두 모델의 출력 shape이 동일 → STEP 2에서 한 줄만 바꾸면 됩니다


## STEP 2 — SimpleBaseline으로 변경하여 훈련하기



In [11]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 2 : SimpleBaseline으로 변경하여 훈련
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#
# 기존 train() 함수와 비교:
#   변경된 줄: model = SimpleBaseline(...)   ← 이것만 다름
#   동일한 부분: DataLoader, Trainer, device, 저장 경로 모두 동일

epochs        = 5        # 최소 3 이상, 권장 5 이상
batch_size    = 16
num_heatmap   = 16
learning_rate = 0.0007

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"학습 디바이스: {device}")

train_loader = create_dataloader(TRAIN_JSON, IMAGE_PATH, batch_size, num_heatmap, is_train=True)
val_loader   = create_dataloader(VALID_JSON, IMAGE_PATH, batch_size, num_heatmap, is_train=False)

if not os.path.exists(MODEL_PATH):
    os.makedirs(MODEL_PATH)

# ─────────────────────────────────────────────────────────
# ★ 변경 포인트: StackedHourglassNetwork → SimpleBaseline
# 기존 코드:
#   model = StackedHourglassNetwork(IMAGE_SHAPE, num_stack=4,
#                                   num_residual=1, num_heatmap=num_heatmap)
# 변경 후:
model = SimpleBaseline(num_heatmap=num_heatmap, pretrained=True)
# ─────────────────────────────────────────────────────────

model.to(device)

sb_trainer = Trainer(
    model,
    epochs,
    batch_size,
    initial_learning_rate=learning_rate
)

print("SimpleBaseline 학습 시작...")
sb_best_model = sb_trainer.run(train_loader, val_loader, device)
print(f"\n✅ SimpleBaseline 학습 완료")
print(f"   저장된 최적 모델: {sb_best_model}")

학습 디바이스: cuda
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /home/jovyan/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 174MB/s]


단일 GPU 혹은 CPU 사용
SimpleBaseline 학습 시작...
Start epoch 1 with learning rate 0.000700


/tmp/ipykernel_1317/1965009459.py:71: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x  = torch.round(torch.tensor(kp_x, dtype=torch.float32) * heatmap_shape[0]).int()
/tmp/ipykernel_1317/1965009459.py:72: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y  = torch.round(torch.tensor(kp_y, dtype=torch.float32) * heatmap_shape[1]).int()
/tmp/ipykernel_1317/1965009459.py:71: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x  = torch.round(torch.tensor(kp_x, dtype=torch.float32) * heatmap_shape[0]).int()
/tmp/ipykernel_1317/1965009459.py:72: UserW

[Train] batch 1 loss 0.5154 avg_loss 0.5154
[Train] batch 2 loss 0.4536 avg_loss 0.4845
[Train] batch 3 loss 0.4905 avg_loss 0.4865
[Train] batch 4 loss 0.4871 avg_loss 0.4866
[Train] batch 5 loss 0.4499 avg_loss 0.4793
[Train] batch 6 loss 0.4392 avg_loss 0.4726
[Train] batch 7 loss 0.4212 avg_loss 0.4653
[Train] batch 8 loss 0.4233 avg_loss 0.4600
[Train] batch 9 loss 0.4160 avg_loss 0.4551
[Train] batch 10 loss 0.4434 avg_loss 0.4540
[Train] batch 11 loss 0.4316 avg_loss 0.4519
[Train] batch 12 loss 0.4181 avg_loss 0.4491
[Train] batch 13 loss 0.4253 avg_loss 0.4473
[Train] batch 14 loss 0.4116 avg_loss 0.4447
[Train] batch 15 loss 0.4313 avg_loss 0.4438
[Train] batch 16 loss 0.4407 avg_loss 0.4436
[Train] batch 17 loss 0.4118 avg_loss 0.4418
[Train] batch 18 loss 0.4189 avg_loss 0.4405
[Train] batch 19 loss 0.4194 avg_loss 0.4394
[Train] batch 20 loss 0.3864 avg_loss 0.4367
[Train] batch 21 loss 0.4156 avg_loss 0.4357
[Train] batch 22 loss 0.4196 avg_loss 0.4350
[Train] batch 23 lo

...

[Val] batch 1 loss 0.2920
[Val] batch 2 loss 0.2706
[Val] batch 3 loss 0.2753
[Val] batch 4 loss 0.2742
[Val] batch 5 loss 0.3198
[Val] batch 6 loss 0.3226
[Val] batch 7 loss 0.2764
[Val] batch 8 loss 0.2860
[Val] batch 9 loss 0.2816
[Val] batch 10 loss 0.2796
[Val] batch 11 loss 0.2975
[Val] batch 12 loss 0.2689
[Val] batch 13 loss 0.3277
[Val] batch 14 loss 0.2625
[Val] batch 15 loss 0.3058
[Val] batch 16 loss 0.2882
[Val] batch 17 loss 0.3043
[Val] batch 18 loss 0.2431
[Val] batch 19 loss 0.3063
[Val] batch 20 loss 0.2671
[Val] batch 21 loss 0.3039
[Val] batch 22 loss 0.2681
[Val] batch 23 loss 0.2783
[Val] batch 24 loss 0.3124
[Val] batch 25 loss 0.2919
[Val] batch 26 loss 0.2609
[Val] batch 27 loss 0.2733
[Val] batch 28 loss 0.2581
[Val] batch 29 loss 0.2852
[Val] batch 30 loss 0.2960
[Val] batch 31 loss 0.2437
[Val] batch 32 loss 0.2920
[Val] batch 33 loss 0.2790
[Val] batch 34 loss 0.3078
[Val] batch 35 loss 0.2905
[Val] batch 36 loss 0.2902
[Val] batch 37 loss 0.2604
[Val] batc

/tmp/ipykernel_1317/1965009459.py:71: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x  = torch.round(torch.tensor(kp_x, dtype=torch.float32) * heatmap_shape[0]).int()
/tmp/ipykernel_1317/1965009459.py:71: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x  = torch.round(torch.tensor(kp_x, dtype=torch.float32) * heatmap_shape[0]).int()
/tmp/ipykernel_1317/1965009459.py:72: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y  = torch.round(torch.tensor(kp_y, dtype=torch.float32) * heatmap_shape[1]).int()
/tmp/ipykernel_1317/1965009459.py:72: UserW

[Train] batch 1 loss 0.2610 avg_loss 0.2610
[Train] batch 2 loss 0.2718 avg_loss 0.2664
[Train] batch 3 loss 0.2714 avg_loss 0.2681
[Train] batch 4 loss 0.2631 avg_loss 0.2668
[Train] batch 5 loss 0.2633 avg_loss 0.2661
[Train] batch 6 loss 0.2778 avg_loss 0.2681
[Train] batch 7 loss 0.2601 avg_loss 0.2669
[Train] batch 8 loss 0.2514 avg_loss 0.2650
[Train] batch 9 loss 0.2806 avg_loss 0.2667
[Train] batch 10 loss 0.2524 avg_loss 0.2653
[Train] batch 11 loss 0.2408 avg_loss 0.2631
[Train] batch 12 loss 0.2875 avg_loss 0.2651
[Train] batch 13 loss 0.2536 avg_loss 0.2642
[Train] batch 14 loss 0.2697 avg_loss 0.2646
[Train] batch 15 loss 0.2706 avg_loss 0.2650
[Train] batch 16 loss 0.2716 avg_loss 0.2654
[Train] batch 17 loss 0.2777 avg_loss 0.2661
[Train] batch 18 loss 0.2556 avg_loss 0.2656
[Train] batch 19 loss 0.2490 avg_loss 0.2647
[Train] batch 20 loss 0.2465 avg_loss 0.2638
[Train] batch 21 loss 0.2667 avg_loss 0.2639
[Train] batch 22 loss 0.2765 avg_loss 0.2645
[Train] batch 23 lo

/tmp/ipykernel_1317/1965009459.py:71: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x  = torch.round(torch.tensor(kp_x, dtype=torch.float32) * heatmap_shape[0]).int()
/tmp/ipykernel_1317/1965009459.py:72: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y  = torch.round(torch.tensor(kp_y, dtype=torch.float32) * heatmap_shape[1]).int()
/tmp/ipykernel_1317/1965009459.py:71: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x  = torch.round(torch.tensor(kp_x, dtype=torch.float32) * heatmap_shape[0]).int()
/tmp/ipykernel_1317/1965009459.py:72: UserW

[Val] batch 1 loss 0.2802
[Val] batch 2 loss 0.2708
[Val] batch 3 loss 0.2736
[Val] batch 4 loss 0.2692
[Val] batch 5 loss 0.3137
[Val] batch 6 loss 0.3168
[Val] batch 7 loss 0.2723
[Val] batch 8 loss 0.2842
[Val] batch 9 loss 0.2783
[Val] batch 10 loss 0.2775
[Val] batch 11 loss 0.2887
[Val] batch 12 loss 0.2629
[Val] batch 13 loss 0.3234
[Val] batch 14 loss 0.2638
[Val] batch 15 loss 0.3080
[Val] batch 16 loss 0.2912
[Val] batch 17 loss 0.3089
[Val] batch 18 loss 0.2382
[Val] batch 19 loss 0.3000
[Val] batch 20 loss 0.2635
[Val] batch 21 loss 0.2975
[Val] batch 22 loss 0.2569
[Val] batch 23 loss 0.2851
[Val] batch 24 loss 0.3036
[Val] batch 25 loss 0.2856
[Val] batch 26 loss 0.2562
[Val] batch 27 loss 0.2805
[Val] batch 28 loss 0.2480
[Val] batch 29 loss 0.2850
[Val] batch 30 loss 0.2977
[Val] batch 31 loss 0.2443
[Val] batch 32 loss 0.2896
[Val] batch 33 loss 0.2771
[Val] batch 34 loss 0.2944
[Val] batch 35 loss 0.2812
[Val] batch 36 loss 0.2789
[Val] batch 37 loss 0.2585
[Val] batc

## STEP 3 — 두 모델의 비교



In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 3-준비 : StackedHourglass 동일 조건으로 학습
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# epochs, batch_size, learning_rate를 STEP 2와 완전히 동일하게 맞춥니다.
# 이미 학습된 HG 모델이 있다면 아래 학습 셀을 건너뛰고
# "기존 가중치 불러오기" 부분을 실행하세요.

hg_model = StackedHourglassNetwork(
    IMAGE_SHAPE, num_stack=4, num_residual=1, num_heatmap=num_heatmap
)
hg_model.to(device)

hg_trainer = Trainer(
    hg_model,
    epochs,           # STEP 2와 동일한 epoch 수
    batch_size,
    initial_learning_rate=learning_rate
)

print("StackedHourglass 학습 시작...")
hg_best_model = hg_trainer.run(train_loader, val_loader, device)
print(f"\n✅ StackedHourglass 학습 완료")
print(f"   저장된 최적 모델: {hg_best_model}")

단일 GPU 혹은 CPU 사용
StackedHourglass 학습 시작...
Start epoch 1 with learning rate 0.000700


/tmp/ipykernel_1317/1965009459.py:71: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x  = torch.round(torch.tensor(kp_x, dtype=torch.float32) * heatmap_shape[0]).int()
/tmp/ipykernel_1317/1965009459.py:72: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y  = torch.round(torch.tensor(kp_y, dtype=torch.float32) * heatmap_shape[1]).int()
/tmp/ipykernel_1317/1965009459.py:71: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x  = torch.round(torch.tensor(kp_x, dtype=torch.float32) * heatmap_shape[0]).int()
/tmp/ipykernel_1317/1965009459.py:72: UserW

[Train] batch 1 loss 2.1821 avg_loss 2.1821
[Train] batch 2 loss 2.1457 avg_loss 2.1639
[Train] batch 3 loss 2.0314 avg_loss 2.1197
[Train] batch 4 loss 2.0193 avg_loss 2.0946
[Train] batch 5 loss 1.9227 avg_loss 2.0603
[Train] batch 6 loss 1.9707 avg_loss 2.0453
[Train] batch 7 loss 1.9256 avg_loss 2.0282
[Train] batch 8 loss 1.8731 avg_loss 2.0088
[Train] batch 9 loss 1.9070 avg_loss 1.9975
[Train] batch 10 loss 1.8017 avg_loss 1.9779
[Train] batch 11 loss 1.8711 avg_loss 1.9682
[Train] batch 12 loss 1.7963 avg_loss 1.9539
[Train] batch 13 loss 1.6594 avg_loss 1.9312
[Train] batch 14 loss 1.8324 avg_loss 1.9242
[Train] batch 15 loss 1.8146 avg_loss 1.9169
[Train] batch 16 loss 1.7801 avg_loss 1.9083
[Train] batch 17 loss 1.7729 avg_loss 1.9004
[Train] batch 18 loss 1.7721 avg_loss 1.8932
[Train] batch 19 loss 1.7993 avg_loss 1.8883
[Train] batch 20 loss 1.6881 avg_loss 1.8783
[Train] batch 21 loss 1.7400 avg_loss 1.8717
[Train] batch 22 loss 1.6129 avg_loss 1.8599
[Train] batch 23 lo

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 3-① : 학습 진행 경과 비교 (loss 감소 현황)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 루브릭 조건: "학습 진행 경과 (loss 감소 현황)" 충족

epochs_range = range(1, epochs + 1)
fig, axes    = plt.subplots(1, 2, figsize=(14, 5))

# ── Train Loss ──
axes[0].plot(epochs_range, hg_trainer.train_losses,
             'b-o', linewidth=2, markersize=7, label='StackedHourglass')
axes[0].plot(epochs_range, sb_trainer.train_losses,
             'r-s', linewidth=2, markersize=7, label='SimpleBaseline')
# 각 점 위에 수치 표기
for i, (hl, sl) in enumerate(zip(hg_trainer.train_losses, sb_trainer.train_losses), 1):
    axes[0].annotate(f'{hl:.3f}', (i, hl), textcoords='offset points',
                     xytext=(0, 8), ha='center', fontsize=8, color='blue')
    axes[0].annotate(f'{sl:.3f}', (i, sl), textcoords='offset points',
                     xytext=(0,-14), ha='center', fontsize=8, color='red')
axes[0].set_title('Train Loss 비교', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss',  fontsize=12)
axes[0].legend(fontsize=12)
axes[0].grid(True, alpha=0.3)

# ── Validation Loss ──
axes[1].plot(epochs_range, hg_trainer.val_losses,
             'b-o', linewidth=2, markersize=7, label='StackedHourglass')
axes[1].plot(epochs_range, sb_trainer.val_losses,
             'r-s', linewidth=2, markersize=7, label='SimpleBaseline')
for i, (hl, sl) in enumerate(zip(hg_trainer.val_losses, sb_trainer.val_losses), 1):
    axes[1].annotate(f'{hl:.3f}', (i, hl), textcoords='offset points',
                     xytext=(0, 8), ha='center', fontsize=8, color='blue')
    axes[1].annotate(f'{sl:.3f}', (i, sl), textcoords='offset points',
                     xytext=(0,-14), ha='center', fontsize=8, color='red')
axes[1].set_title('Validation Loss 비교', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Loss',  fontsize=12)
axes[1].legend(fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.suptitle(
    f'StackedHourglass vs SimpleBaseline  |  {epochs} epochs, batch={batch_size}',
    fontsize=15
)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_PATH, 'loss_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

# ── 수치 요약 출력 ──
print("=" * 65)
print("  학습 결과 수치 요약")
print("=" * 65)
print(f"{'':30} {'StackedHourglass':>16} {'SimpleBaseline':>16}")
print("-" * 65)
for ep in range(epochs):
    print(f"  Epoch {ep+1:2d} Train Loss          "
          f"{hg_trainer.train_losses[ep]:>16.4f} "
          f"{sb_trainer.train_losses[ep]:>16.4f}")
    print(f"  Epoch {ep+1:2d} Val   Loss          "
          f"{hg_trainer.val_losses[ep]:>16.4f} "
          f"{sb_trainer.val_losses[ep]:>16.4f}")
    print()
print("-" * 65)
print(f"  Best Val Loss               "
      f"{hg_trainer.lowest_val_loss:>16.4f} "
      f"{sb_trainer.lowest_val_loss:>16.4f}")
print("=" * 65)

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 3-② : 최적 가중치 로드
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def load_best_model(model, weights_path, device):
    """저장된 가중치를 불러와 eval 모드로 전환합니다."""
    ckpt = torch.load(weights_path, map_location=device)
    # DataParallel로 저장된 경우 'module.' prefix 제거
    ckpt = {k.replace('module.', ''): v for k, v in ckpt.items()}
    model.load_state_dict(ckpt)
    model.eval()
    return model

hg_eval = StackedHourglassNetwork(IMAGE_SHAPE, num_stack=4, num_residual=1, num_heatmap=num_heatmap)
hg_eval.to(device)
hg_eval = load_best_model(hg_eval, hg_best_model, device)

sb_eval = SimpleBaseline(num_heatmap=num_heatmap, pretrained=False)
sb_eval.to(device)
sb_eval = load_best_model(sb_eval, sb_best_model, device)

print(f"StackedHourglass 가중치 로드 완료: {hg_best_model}")
print(f"SimpleBaseline   가중치 로드 완료: {sb_best_model}")

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 3-③ : Pose Estimation 결과 시각화 (정성적 비교)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 루브릭 조건: "Pose Estimation 결과 시각화 (정성적 비교)" 충족
#
# 테스트 이미지를 두 모델에 통과시켜 skeleton을 나란히 그립니다.

# 테스트 이미지 다운로드
!wget -q "https://images.unsplash.com/photo-1546427660-eb346c344ba5?w=800" \
     -O ~/work/mpii/test_image.jpg

test_image_path = os.path.join(PROJECT_PATH, 'test_image.jpg')

# ── 두 모델 나란히 비교 ──
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

for ax, (label, model) in zip(axes, [
    (f'StackedHourglass\n(best val loss: {hg_trainer.lowest_val_loss:.4f})', hg_eval),
    (f'SimpleBaseline\n(best val loss: {sb_trainer.lowest_val_loss:.4f})',   sb_eval),
]):
    image, kps = predict(model, test_image_path)
    img_np     = np.array(image)
    h, w, _    = img_np.shape

    ax.imshow(img_np)
    joints = [(kp[0].item()*w, kp[1].item()*h) for kp in kps]
    colors = plt.cm.hsv(np.linspace(0, 0.85, len(MPII_BONES)))

    for bone, color in zip(MPII_BONES, colors):
        j1, j2 = joints[bone[0]], joints[bone[1]]
        ax.plot([j1[0], j2[0]], [j1[1], j2[1]], '-',
                color=color, linewidth=4, alpha=0.85)
    for jx, jy in joints:
        ax.scatter(jx, jy, s=60, c='white', edgecolors='black', linewidths=1.5, zorder=5)

    ax.set_title(label, fontsize=13, fontweight='bold')
    ax.axis('off')

plt.suptitle('Pose Estimation 결과 비교 (정성적)', fontsize=16)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_PATH, 'pose_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"이미지 저장 완료: {os.path.join(PROJECT_PATH, 'pose_comparison.png')}")

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 3-④ : Validation 데이터셋 샘플로 추가 정성적 비교
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 외부 이미지 1장 외에 실제 MPII validation 샘플 3장으로도 비교합니다.

NUM_SAMPLES    = 3
imgs_batch, _  = next(iter(val_loader))
imgs_batch     = imgs_batch[:NUM_SAMPLES]

fig, axes = plt.subplots(NUM_SAMPLES, 2, figsize=(12, 5 * NUM_SAMPLES))

for row in range(NUM_SAMPLES):
    img_t  = imgs_batch[row]   # (C, H, W), [-1, 1]
    # [-1,1] → [0,255] 로 역정규화해서 시각화
    img_np = ((img_t.permute(1,2,0).numpy() + 1) * 127.5).clip(0,255).astype(np.uint8)
    h, w   = img_np.shape[:2]

    for col, (label, model) in enumerate([
        ('StackedHourglass', hg_eval),
        ('SimpleBaseline',   sb_eval),
    ]):
        model.eval()
        inp = img_t.unsqueeze(0).to(device)
        with torch.no_grad():
            outputs = model(inp)
        hm  = outputs[-1].squeeze(0).permute(1,2,0).detach().cpu()
        kps = extract_keypoints_from_heatmap(hm)

        ax     = axes[row, col]
        ax.imshow(img_np)
        joints = [(kp[0].item()*w, kp[1].item()*h) for kp in kps]
        for bone in MPII_BONES:
            j1, j2 = joints[bone[0]], joints[bone[1]]
            ax.plot([j1[0], j2[0]], [j1[1], j2[1]], 'c-', linewidth=3, alpha=0.85)
        for jx, jy in joints:
            ax.scatter(jx, jy, s=40, c='yellow', edgecolors='black', linewidths=1, zorder=5)

        ax.set_title(f'{label}  |  Val 샘플 {row+1}', fontsize=11)
        ax.axis('off')

plt.suptitle('Validation 샘플 Pose Estimation 비교', fontsize=15)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_PATH, 'val_pose_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"이미지 저장 완료: {os.path.join(PROJECT_PATH, 'val_pose_comparison.png')}")

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 3-⑤ : 최종 비교 요약 출력
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

hg_params = sum(p.numel() for p in hg_eval.parameters())
sb_params = sum(p.numel() for p in sb_eval.parameters())

print("\n" + "=" * 62)
print("         최종 비교 요약")
print("=" * 62)
print(f"{'항목':<28} {'StackedHourglass':>17} {'SimpleBaseline':>15}")
print("-" * 62)
print(f"{'파라미터 수':<28} {hg_params:>17,} {sb_params:>15,}")
print(f"{'ImageNet pretrained':<28} {'❌':>17} {'✅':>15}")
print(f"{'출력 heatmap 수 (num_stack)':<28} {4:>17} {1:>15}")
print(f"{'학습 Epoch':<28} {epochs:>17} {epochs:>15}")
print(f"{'Best Validation Loss':<28} {hg_trainer.lowest_val_loss:>17.4f} {sb_trainer.lowest_val_loss:>15.4f}")
print(f"{'최종 Train Loss':<28} {hg_trainer.train_losses[-1]:>17.4f} {sb_trainer.train_losses[-1]:>15.4f}")
print(f"{'최종 Val   Loss':<28} {hg_trainer.val_losses[-1]:>17.4f} {sb_trainer.val_losses[-1]:>15.4f}")
print("=" * 62)

if sb_trainer.lowest_val_loss < hg_trainer.lowest_val_loss:
    diff = hg_trainer.lowest_val_loss - sb_trainer.lowest_val_loss
    print(f"\n🏆 SimpleBaseline이 {diff:.4f} 낮은 validation loss 달성")
    print("   → 더 단순한 구조 + pretrained backbone으로 더 좋은 성능")
else:
    diff = sb_trainer.lowest_val_loss - hg_trainer.lowest_val_loss
    print(f"\n🏆 StackedHourglass가 {diff:.4f} 낮은 validation loss 달성")
    print("   → 더 많은 epoch 또는 더 긴 학습이 필요할 수 있음")

print("""
분석 포인트
───────────────────────────────────────────────────────────
[SimpleBaseline의 강점]
  - ResNet-50 pretrained → 초기 수렴 빠름
  - 구조 단순 → 코드 짧고 유지보수 쉬움
  - 적은 epoch에서도 좋은 성능

[StackedHourglass의 강점]
  - Intermediate supervision (num_stack=4) → 점진적 refinement
  - Skip connection → 고해상도 세부 정보 보존
  - epoch을 충분히 주면 더 세밀한 예측 가능
""")

# 실험
## 실험 A: Deconv Head가 해상도를 복원하는 과정 시각화

In [ ]:
# ── 실험 A : SimpleBaseline Deconv 단계별 feature map ─────
# 핵심 질문: 8×8 feature map이 어떻게 64×64 heatmap으로 복원되는가?
# forward hook으로 중간 출력을 캡처합니다.

feature_maps = {}

def make_hook(name):
    def hook(module, input, output):
        feature_maps[name] = output.detach().cpu()
    return hook

# deconv_head의 각 ReLU 출력에 hook 등록
hooks     = []
step_idx  = 0
for layer in sb_eval.deconv_head:
    if isinstance(layer, nn.ReLU):
        h = layer.register_forward_hook(make_hook(f'deconv_step{step_idx+1}'))
        hooks.append(h);  step_idx += 1

test_img = Image.open(test_image_path).convert('RGB')
pre      = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x*2 - 1)
])
inp = pre(test_img).unsqueeze(0).to(device)

with torch.no_grad():
    _ = sb_eval(inp)

for h in hooks:
    h.remove()

# 시각화
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(np.array(test_img.resize((256,256))))
axes[0].set_title('입력\n256×256', fontsize=12);  axes[0].axis('off')

step_titles = ['Deconv 1단계\n8×8 → 16×16', 'Deconv 2단계\n16×16 → 32×32', 'Deconv 3단계\n32×32 → 64×64']
for i in range(3):
    fm = feature_maps[f'deconv_step{i+1}'][0, 0].numpy()
    axes[i+1].imshow(fm, cmap='viridis')
    axes[i+1].set_title(step_titles[i], fontsize=12)
    axes[i+1].axis('off')
    print(f"deconv_step{i+1} feature map shape: {feature_maps[f'deconv_step{i+1}'].shape}")

plt.suptitle('SimpleBaseline: Deconv Head 해상도 복원 과정', fontsize=14)
plt.tight_layout()
plt.show()

## 실험 B: pretrained=True vs False 초기 수렴 속도 비교


In [ ]:
# ── 실험 B : pretrained 효과를 1 epoch으로 빠르게 확인 ────
# pretrained backbone이 없으면 초기 loss가 얼마나 높은지 확인합니다.

results_pretrain = {}
for use_pretrained in [True, False]:
    tag   = "pretrained=True " if use_pretrained else "pretrained=False"
    model = SimpleBaseline(num_heatmap=16, pretrained=use_pretrained).to(device)
    tr    = Trainer(model, epochs=1, global_batch_size=batch_size,
                    initial_learning_rate=learning_rate)
    tr.run(train_loader, val_loader, device)
    results_pretrain[tag] = {
        'train': tr.train_losses[0],
        'val'  : tr.val_losses[0],
    }
    del model;  torch.cuda.empty_cache()

print("\n" + "=" * 52)
print("  실험 B: pretrained 여부별 1 Epoch 결과")
print("=" * 52)
print(f"{'':22} {'Train Loss':>12} {'Val Loss':>12}")
print("-" * 52)
for tag, vals in results_pretrain.items():
    print(f"{tag:<22} {vals['train']:>12.4f} {vals['val']:>12.4f}")
print("=" * 52)
print("\n📌 pretrained=True의 Train/Val Loss가 낮으면")
print("   → ImageNet 사전학습이 포즈 추정에도 유효한 피처를 제공한다는 증거")

## 실험 C: num_stack 수에 따른 HG 파라미터 vs 성능 트레이드오프

In [ ]:
# ── 실험 C : num_stack=1 vs 4 파라미터 / 추론속도 비교 ────
# StackedHourglass에서 "stack을 쌓는 것"이 얼마나 비용이 드는지 측정합니다.

import time

device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dummy   = torch.randn(4, 3, 256, 256).to(device)

print("=" * 62)
print("  num_stack에 따른 파라미터 / 추론 속도 비교 (배치=4)")
print("=" * 62)
print(f"{'모델':<22} {'파라미터(M)':>12} {'추론(ms)':>10} {'출력 수':>8}")
print("-" * 62)

for num_stack in [1, 2, 4]:
    model = StackedHourglassNetwork(num_stack=num_stack, num_heatmap=16).to(device)
    model.eval()
    params = sum(p.numel() for p in model.parameters()) / 1e6
    with torch.no_grad():
        _ = model(dummy)
        t0 = time.time()
        for _ in range(10):
            out = model(dummy)
        ms = (time.time() - t0) / 10 * 1000
    print(f"  HG num_stack={num_stack}       {params:>10.1f}M {ms:>10.1f}ms {len(out):>8}")
    del model

sb_tmp = SimpleBaseline(num_heatmap=16, pretrained=False).to(device)
sb_tmp.eval()
sb_p = sum(p.numel() for p in sb_tmp.parameters()) / 1e6
with torch.no_grad():
    _ = sb_tmp(dummy)
    t0 = time.time()
    for _ in range(10):
        out_sb = sb_tmp(dummy)
    ms_sb = (time.time() - t0) / 10 * 1000
print(f"  SimpleBaseline        {sb_p:>10.1f}M {ms_sb:>10.1f}ms {len(out_sb):>8}")
print("=" * 62)
print("\n📌 SimpleBaseline은 num_stack=1 HG보다 파라미터가")
print("   적거나 비슷하지만, pretrained 덕분에 성능이 높을 수 있음")

del sb_tmp, dummy
torch.cuda.empty_cache()